Menyiapkan Dataset
-

In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


Membuat SparkSession
-

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    sum as spark_sum,
    when,
    round as spark_round
)

spark = SparkSession.builder \
    .appName("Tugas6-ETL") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap.")
print("Versi Spark:", spark.version)

26/10/10 17:22:12 WARN Utils: Your hostname, zaydann resolves to a loopback address: 127.0.1.1; using 10.46.162.1 instead (on interface wlo1)
26/10/10 17:22:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/10 17:22:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap.
Versi Spark: 3.5.9


A. Extract
-

Membaca Ketiga Sumber Data

In [3]:
# Membaca data transaksi
df_transaksi_tugas6 = spark.read.csv(
    "tugas6_transaksi.csv",
    header=True,
    inferSchema=True
)

# Membaca data produk berformat JSON
df_produk_tugas6 = spark.read.json(
    "tugas6_produk.json"
)

# Membaca data ulasan
df_ulasan_tugas6 = spark.read.csv(
    "tugas6_ulasan.csv",
    header=True,
    inferSchema=True
)

# Menampilkan jumlah baris setiap DataFrame
print("Jumlah transaksi:", df_transaksi_tugas6.count())
print("Jumlah produk:", df_produk_tugas6.count())
print("Jumlah ulasan:", df_ulasan_tugas6.count())

# Menampilkan skema masing-masing DataFrame
print("\n=== Skema Data Transaksi ===")
df_transaksi_tugas6.printSchema()

print("\n=== Skema Data Produk ===")
df_produk_tugas6.printSchema()

print("\n=== Skema Data Ulasan ===")
df_ulasan_tugas6.printSchema()

Jumlah transaksi: 5000
Jumlah produk: 30
Jumlah ulasan: 3500

=== Skema Data Transaksi ===
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)


=== Skema Data Produk ===
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)


=== Skema Data Ulasan ===
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



B. Transform — Penggabungan
-

Menggabungkan Ketiga DataFrame

In [4]:
df_gabungan_tugas6 = df_transaksi_tugas6.join(
    df_produk_tugas6,
    on="product_id",
    how="inner"
).join(
    df_ulasan_tugas6,
    on="order_id",
    how="left"
)

# Menambahkan kolom total_pendapatan
df_gabungan_tugas6 = df_gabungan_tugas6.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga")
)

# Menampilkan hasil penggabungan
df_gabungan_tugas6.show(10)

# Memeriksa jumlah baris
print(
    "Jumlah data setelah join:",
    df_gabungan_tugas6.count()
)

+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|order_id|product_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|rating|total_pendapatan|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|     TX0|        21|           2|2026-10-10 00:00:00| 25000|  Elektronik|  Produk-21|     1|           50000|
|     TX1|         8|           6|2026-10-25 00:00:00|250000|     Fashion|   Produk-8|  NULL|         1500000|
|     TX2|        25|           4|2026-10-13 00:00:00| 25000|  Elektronik|  Produk-25|     2|          100000|
|     TX3|         3|           4|2026-10-18 00:00:00| 75000|     Fashion|   Produk-3|     5|          300000|
|     TX4|        19|           6|2026-10-07 00:00:00| 75000|Rumah Tangga|  Produk-19|  NULL|          450000|
|     TX5|        10|           3|2026-10-01 00:00:00| 75000|  Elektronik|  Produk-10|     5|          225000|
|

C. Transform — Penanganan Data Kosong dan Pengayaan
-

Membuat Kolom ada_ulasan dan Mengisi Rating Kosong

In [5]:
# Menentukan apakah transaksi memiliki ulasan
df_gabungan_tugas6 = df_gabungan_tugas6.withColumn(
    "ada_ulasan",
    col("rating").isNotNull()
)

# Mengisi rating yang kosong dengan angka 0
df_gabungan_tugas6 = df_gabungan_tugas6.na.fill({
    "rating": 0
})

# Menampilkan hasil setelah penanganan data kosong
df_gabungan_tugas6.select(
    "order_id",
    "product_id",
    "nama_produk",
    "kategori",
    "rating",
    "ada_ulasan",
    "total_pendapatan"
).show(10)

# Menghitung transaksi yang memiliki ulasan
print(
    "Transaksi dengan ulasan:",
    df_gabungan_tugas6.filter(
        col("ada_ulasan") == True
    ).count()
)

# Menghitung transaksi yang tidak memiliki ulasan
print(
    "Transaksi tanpa ulasan:",
    df_gabungan_tugas6.filter(
        col("ada_ulasan") == False
    ).count()
)

+--------+----------+-----------+------------+------+----------+----------------+
|order_id|product_id|nama_produk|    kategori|rating|ada_ulasan|total_pendapatan|
+--------+----------+-----------+------------+------+----------+----------------+
|     TX0|        21|  Produk-21|  Elektronik|     1|      true|           50000|
|     TX1|         8|   Produk-8|     Fashion|     0|     false|         1500000|
|     TX2|        25|  Produk-25|  Elektronik|     2|      true|          100000|
|     TX3|         3|   Produk-3|     Fashion|     5|      true|          300000|
|     TX4|        19|  Produk-19|Rumah Tangga|     0|     false|          450000|
|     TX5|        10|  Produk-10|  Elektronik|     5|      true|          225000|
|     TX6|        26|  Produk-26|   Kesehatan|     4|      true|          450000|
|     TX7|         4|   Produk-4|     Makanan|     0|     false|          150000|
|     TX8|         7|   Produk-7|  Elektronik|     5|      true|          150000|
|     TX9|      

D. Load
-

Menyimpan Hasil ETL ke HDFS dalam Format Parquet

In [6]:
# Membuat direktori tujuan
!hdfs dfs -mkdir -p /user/zayy/tugas6

# Menentukan path penyimpanan
path_hasil_etl = "hdfs://localhost:9000/user/zayy/tugas6/hasil_etl"

# Menyimpan data ke Parquet dengan partitioning
df_gabungan_tugas6.write \
    .mode("overwrite") \
    .partitionBy("kategori") \
    .parquet(path_hasil_etl)

print("Data ETL berhasil disimpan ke HDFS.")

[Stage 37:>                                                         (0 + 1) / 1]

Data ETL berhasil disimpan ke HDFS.


Memeriksa Hasil Penyimpanan di HDFS

In [7]:
!hdfs dfs -ls -R /user/zayy/tugas6/hasil_etl

-rw-r--r--   3 zaydann supergroup          0 2026-10-10 17:07 /user/zayy/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - zaydann supergroup          0 2026-10-10 17:07 /user/zayy/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 zaydann supergroup      16364 2026-10-10 17:07 /user/zayy/tugas6/hasil_etl/kategori=Elektronik/part-00000-0372f2f2-928c-4470-af54-61d59ce47861.c000.snappy.parquet
drwxr-xr-x   - zaydann supergroup          0 2026-10-10 17:07 /user/zayy/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 zaydann supergroup      11134 2026-10-10 17:07 /user/zayy/tugas6/hasil_etl/kategori=Fashion/part-00000-0372f2f2-928c-4470-af54-61d59ce47861.c000.snappy.parquet
drwxr-xr-x   - zaydann supergroup          0 2026-10-10 17:07 /user/zayy/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 zaydann supergroup      10393 2026-10-10 17:07 /user/zayy/tugas6/hasil_etl/kategori=Kesehatan/part-00000-0372f2f2-928c-4470-af54-61d59ce47861.c000.snappy.parquet
drwxr-xr-x   - zaydann supergroup          0 

Membaca Kembali dan Memeriksa Jumlah Data

In [8]:
# Membaca kembali hasil ETL dari HDFS
df_hasil_etl = spark.read.parquet(
    "hdfs://localhost:9000/user/zayy/tugas6/hasil_etl"
)

# Memeriksa jumlah baris
print(
    "Jumlah baris setelah dibaca kembali:",
    df_hasil_etl.count()
)

# Menampilkan contoh data
df_hasil_etl.show(10)

Jumlah baris setelah dibaca kembali: 5000
+--------+----------+------------+-------------------+------+-----------+------+----------------+----------+----------+
|order_id|product_id|unit_terjual|            tanggal| harga|nama_produk|rating|total_pendapatan|ada_ulasan|  kategori|
+--------+----------+------------+-------------------+------+-----------+------+----------------+----------+----------+
|     TX0|        21|           2|2026-10-10 00:00:00| 25000|  Produk-21|     1|           50000|      true|Elektronik|
|     TX2|        25|           4|2026-10-13 00:00:00| 25000|  Produk-25|     2|          100000|      true|Elektronik|
|     TX5|        10|           3|2026-10-01 00:00:00| 75000|  Produk-10|     5|          225000|      true|Elektronik|
|     TX8|         7|           2|2026-10-09 00:00:00| 75000|   Produk-7|     5|          150000|      true|Elektronik|
|     TX9|        30|           1|2026-10-27 00:00:00|250000|  Produk-30|     0|          250000|     false|Elektronik

E. Insight Akhir
-

Menghitung Persentase Transaksi yang Memiliki Ulasan

In [9]:
insight_kategori = df_hasil_etl.groupBy(
    "kategori"
).agg(
    count("order_id").alias("total_transaksi"),
    spark_sum(
        when(col("ada_ulasan") == True, 1).otherwise(0)
    ).alias("transaksi_dengan_ulasan")
)

# Menghitung persentase transaksi dengan ulasan
insight_kategori = insight_kategori.withColumn(
    "persentase_ulasan",
    spark_round(
        (
            col("transaksi_dengan_ulasan")
            / col("total_transaksi")
        ) * 100,
        2
    )
)

# Mengurutkan kategori dari persentase terendah
insight_kategori = insight_kategori.orderBy(
    col("persentase_ulasan").asc()
)

# Menampilkan hasil analisis
insight_kategori.show(truncate=False)

# Menampilkan kategori dengan persentase terendah
kategori_terendah = insight_kategori.first()

print(
    "Kategori dengan persentase ulasan terendah:",
    kategori_terendah["kategori"]
)

print(
    "Persentase ulasan:",
    kategori_terendah["persentase_ulasan"],
    "%"
)

+------------+---------------+-----------------------+-----------------+
|kategori    |total_transaksi|transaksi_dengan_ulasan|persentase_ulasan|
+------------+---------------+-----------------------+-----------------+
|Makanan     |536            |369                    |68.84            |
|Kesehatan   |961            |662                    |68.89            |
|Fashion     |1035           |726                    |70.14            |
|Rumah Tangga|815            |575                    |70.55            |
|Elektronik  |1653           |1168                   |70.66            |
+------------+---------------+-----------------------+-----------------+

Kategori dengan persentase ulasan terendah: Makanan
Persentase ulasan: 68.84 %


Berdasarkan hasil analisis, kategori Makanan memiliki persentase transaksi dengan ulasan paling rendah, yaitu sekitar 68,84%. Hal ini penting diketahui oleh tim marketing karena jumlah ulasan dapat membantu perusahaan memahami pengalaman dan kepuasan pelanggan terhadap produk yang dibeli. Oleh karena itu, tim marketing dapat mendorong pelanggan untuk memberikan ulasan setelah melakukan pembelian, misalnya melalui pengingat atau program apresiasi yang sesuai.

Menutup SparkSession
-

In [3]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
